# Job Packaging & Execution

Before a job can run remotely, it must be packaged, transferred, and executed. These three steps look simple but contain most of the production complexity: handling large projects efficiently without resending unchanged files, injecting runtime parameters without modifying source files, and running notebooks as reproducible, parameterized artifacts. This notebook builds each layer — packaging strategies, runner abstractions, parameter injection, and the SSH execution loop — from first principles.

## Packaging Strategies

**Why packaging matters.** Reproducibility requires that the remote machine runs exactly the code we intend — not a stale version, not a partially-synced directory tree. Packaging also creates an atomic unit: the job either arrives complete or not at all, which makes failure recovery straightforward. Finally, a versioned archive gives us a record of what was submitted.

We use two complementary strategies depending on the use case:

**`tar.gz` approach.** The entire project directory is compressed into a single `.tar.gz` archive and transferred in one SCP call. This is fast for small jobs (under ~50 MB), is self-contained, and works with any SSH endpoint. The downside is that even a one-line change requires retransferring the whole archive.

**`rsync` approach.** `rsync` computes a rolling checksum of source and destination files and transfers only the delta. This is the right choice for large projects or iterative development workflows where you are running many short experiments against the same codebase. The drawback is that `rsync` must be installed on the remote host.

| Strategy | Best for | Transfer size | Remote requirement |
|----------|----------|--------------|--------------------|
| `tar.gz` | One-shot runs, small projects, cold machines | Full archive each run | None beyond SSH |
| `rsync`  | Iterative dev, large projects, `--watch` mode | Delta only | `rsync` on remote |

We implement the `tar.gz` strategy using the stdlib `tarfile` module:

In [ ]:
import pathlib
import tarfile

EXCLUDE_PATTERNS = {
    ".git", "__pycache__", ".ipynb_checkpoints",
    ".mypy_cache", ".ruff_cache",
}


def create_archive(src: pathlib.Path, output: pathlib.Path) -> pathlib.Path:
    """Create a .tar.gz of src, excluding common noise directories."""
    output = output.with_suffix("").with_suffix(".tar.gz")
    with tarfile.open(output, "w:gz") as tar:
        for item in sorted(src.rglob("*")):
            if any(part in EXCLUDE_PATTERNS for part in item.parts):
                continue
            if item.suffix == ".pyc":
                continue
            tar.add(item, arcname=item.relative_to(src.parent))
    return output

Creating an archive from a sample directory and inspecting its contents:

In [ ]:
import tempfile

with tempfile.TemporaryDirectory() as tmp:
    root = pathlib.Path(tmp) / "my_project"
    (root / "src").mkdir(parents=True)
    (root / "src" / "train.py").write_text("print('training...')")
    (root / "src" / "__pycache__").mkdir()     # should be excluded
    (root / "config.yaml").write_text("lr: 3e-4")
    (root / ".git").mkdir()                    # should be excluded

    archive = create_archive(root, pathlib.Path(tmp) / "job")

    with tarfile.open(archive) as tar:
        names = tar.getnames()

print("Archive contents:")
for n in sorted(names):
    print(" ", n)

## The Runner Abstraction

Three runner types correspond to the three `JobType` values. Each runner knows how to translate a `JobSpec` into a shell command for a specific execution modality. This is the single place where execution strategy is decided — the SSH layer that follows is generic and does not know or care which runner produced the command.

We define the shared result type and abstract base first:

In [ ]:
from __future__ import annotations

import abc
import uuid
from dataclasses import dataclass
from enum import Enum
from typing import Any

from pydantic import BaseModel, Field


class JobType(str, Enum):
    notebook = "notebook"
    script   = "script"
    project  = "project"


class JobSpec(BaseModel):
    id:         str      = Field(default_factory=lambda: uuid.uuid4().hex[:8])
    type:       JobType
    entrypoint: str
    path:       str      = "."
    params:     dict[str, Any] = {}
    env:        dict[str, str] = {}


@dataclass
class RunResult:
    job_id:     str
    returncode: int
    stdout:     str
    stderr:     str
    duration_s: float


class BaseRunner(abc.ABC):
    @abc.abstractmethod
    def build_command(self, spec: JobSpec) -> list[str]:
        """Return the shell command as an argv list."""

The three concrete runners and the factory dispatch:

In [ ]:
class ScriptRunner(BaseRunner):
    """Runs a plain Python script: python {entrypoint} --key value ..."""

    def build_command(self, spec: JobSpec) -> list[str]:
        cmd = ["python", spec.entrypoint]
        for key, val in spec.params.items():
            cmd += [f"--{key}", str(val)]
        return cmd


class NotebookRunner(BaseRunner):
    """Runs a notebook via papermill with -p key value flags."""

    def build_command(self, spec: JobSpec) -> list[str]:
        output_nb = f"output_{spec.id}.ipynb"
        cmd = ["papermill", spec.entrypoint, output_nb]
        for key, val in spec.params.items():
            cmd += ["-p", key, str(val)]
        return cmd


class ProjectRunner(BaseRunner):
    """Packages the project as tar.gz, transfers via SCP, then delegates to ScriptRunner."""

    def build_command(self, spec: JobSpec) -> list[str]:
        script_spec = spec.model_copy(update={"type": JobType.script})
        return ScriptRunner().build_command(script_spec)


class RunnerFactory:
    _map: dict[JobType, type[BaseRunner]] = {
        JobType.script:   ScriptRunner,
        JobType.notebook: NotebookRunner,
        JobType.project:  ProjectRunner,
    }

    @classmethod
    def get(cls, job_type: JobType) -> BaseRunner:
        return cls._map[job_type]()

Verifying command construction for all three types:

In [ ]:
for jtype, entry in [
    (JobType.script,   "train.py"),
    (JobType.notebook, "train.ipynb"),
    (JobType.project,  "src/main.py"),
]:
    spec = JobSpec(type=jtype, entrypoint=entry, params={"epochs": 10, "lr": 3e-4})
    cmd  = RunnerFactory.get(jtype).build_command(spec)
    print(f"{jtype.value:10s}  ->  {' '.join(cmd)}")

## Parameter & Environment Injection

Parameters reach the running process via two channels depending on job type:

- **Script params** — `--key value` CLI arguments, parsed by `argparse` or `click` in the script.
- **Notebook params** — `-p key value` papermill flags; papermill injects a new cell after the tagged `# parameters` cell, overriding those variable names.
- **Environment variables** — `{**os.environ, **job_spec.env}` passed as the `env` kwarg to `asyncio.create_subprocess_exec`, so the subprocess inherits the host environment with job-specific overrides layered on top.

We consolidate parameter building into a single utility:

In [ ]:
import os


def build_env(spec: JobSpec) -> dict[str, str]:
    """Merge host environment with job-specific overrides."""
    return {**os.environ, **spec.env}


def params_to_cli_args(params: dict[str, Any]) -> list[str]:
    """Convert params dict to --key value CLI argument list."""
    args: list[str] = []
    for key, val in params.items():
        args += [f"--{key}", str(val)]
    return args


def params_to_papermill_flags(params: dict[str, Any]) -> list[str]:
    """Convert params dict to -p key value papermill flag list."""
    flags: list[str] = []
    for key, val in params.items():
        flags += ["-p", key, str(val)]
    return flags

Checking output for a representative job spec:

In [ ]:
spec = JobSpec(
    type=JobType.notebook,
    entrypoint="train.ipynb",
    params={"epochs": 20, "lr": 3e-4, "model": "resnet50"},
    env={"CUDA_VISIBLE_DEVICES": "0", "WANDB_PROJECT": "nbx-runs"},
)

print("CLI args:     ", params_to_cli_args(spec.params))
print("Papermill:    ", params_to_papermill_flags(spec.params))
print("CUDA visible: ", build_env(spec).get("CUDA_VISIBLE_DEVICES"))
print("WANDB project:", build_env(spec).get("WANDB_PROJECT"))

:::{.callout-note}
Environment injection via `{**os.environ, **spec.env}` means job-specific values override host values. This allows the platform to inject `WANDB_API_KEY` from a secrets store at dispatch time, so that neither the job author nor the source code ever touches secrets directly.

:::

## SSH Remote Execution

The SSH execution layer has four steps: (1) upload the archive via SCP, (2) extract it into a workspace directory on the remote host, (3) run the command built by the runner, and (4) stream stdout and stderr line by line back to the caller.

**Connection reuse.** All four steps share a single `asyncssh.connect(...)` context so that key authentication happens once per job, not four times. `asyncssh` uses the OpenSSH agent if one is running, or a key file from `NbxSettings.ssh_key_path`.

**Workspace isolation.** Each job runs in `/workspace/{spec.id}` — a directory named by the job UUID. This guarantees that concurrent jobs on the same machine never collide on intermediate files and that the artifacts of different runs are trivially separable. The workspace is not cleaned up automatically; a periodic maintenance coroutine should prune workspaces older than $N$ days to avoid filling the remote disk.

**Why SCP over rsync.** SCP transfers the archive in one call and requires no remote rsync binary. For archives under 100 MB, transfer overhead is negligible. `rsync` is preferable for large projects with many unchanged files, but compressing only the changed sources into a fresh archive achieves the same effect at the cost of always uploading a full snapshot — which is acceptable given the small project sizes this platform targets.

The full implementation uses `asyncssh` and is shown as a static block because it requires SSH credentials to run:

```python
# providers/ssh.py — run as a standalone script
import asyncio
import pathlib
import time

import asyncssh


async def execute_remote(
    host: str,
    key_path: str,
    spec: JobSpec,
    archive: pathlib.Path,
) -> RunResult:
    workspace      = f"/workspace/{spec.id}"
    stdout_lines:  list[str] = []
    stderr_lines:  list[str] = []

    async with asyncssh.connect(host, client_keys=[key_path], known_hosts=None) as conn:
        # 1. Upload archive
        remote_archive = f"/tmp/{spec.id}.tar.gz"
        await asyncssh.scp(str(archive), (conn, remote_archive))

        # 2. Extract
        await conn.run(
            f"mkdir -p {workspace} && "
            f"tar -xzf {remote_archive} -C {workspace} --strip-components=1",
            check=True,
        )

        # 3. Build and run command
        runner  = RunnerFactory.get(spec.type)
        cmd_str = " ".join(runner.build_command(spec))
        env_str = " ".join(f"{k}={v}" for k, v in spec.env.items())
        full    = f"cd {workspace} && {env_str + ' ' if env_str else ''}{cmd_str}"

        t0   = time.perf_counter()
        proc = await conn.create_process(full)

        # 4. Stream stdout and stderr concurrently
        async def drain(stream, bucket: list[str]):
            async for line in stream:
                bucket.append(line)
                print(line, end="")  # real impl: push to ConnectionManager

        await asyncio.gather(
            drain(proc.stdout, stdout_lines),
            drain(proc.stderr, stderr_lines),
        )
        returncode = proc.returncode

    return RunResult(
        job_id=spec.id,
        returncode=returncode,
        stdout="".join(stdout_lines),
        stderr="".join(stderr_lines),
        duration_s=time.perf_counter() - t0,
    )
```

We demonstrate the streaming pattern locally — no SSH required — using `asyncio.create_subprocess_shell`:

In [ ]:
import asyncio
import time


async def stream_local(command: str) -> RunResult:
    """Run a shell command locally and stream output line by line."""
    proc = await asyncio.create_subprocess_shell(
        command,
        stdout=asyncio.subprocess.PIPE,
        stderr=asyncio.subprocess.PIPE,
    )

    lines: list[str] = []

    async def drain(stream, tag: str):
        async for raw in stream:
            line = raw.decode().rstrip()
            lines.append(line)
            print(f"[{tag}] {line}")

    t0 = time.perf_counter()
    await asyncio.gather(drain(proc.stdout, "out"), drain(proc.stderr, "err"))
    rc = await proc.wait()

    return RunResult(
        job_id="local-test",
        returncode=rc,
        stdout="\n".join(lines),
        stderr="",
        duration_s=time.perf_counter() - t0,
    )


result = asyncio.run(
    stream_local(
        'python -c "import time; [print(f\'step {i}\') or time.sleep(0.1) for i in range(5)]"'
    )
)
print(f"\nreturncode={result.returncode}  duration={result.duration_s:.2f}s")

:::{.callout-caution}
Reading stdout then stderr (or vice versa) sequentially will deadlock if the process writes enough to fill the OS pipe buffer on the unread stream. Always read both streams concurrently with `asyncio.gather`.

:::

## Appendix: Papermill Deep Dive {#sec-papermill}

**papermill** executes a Jupyter notebook as a script, injecting parameters into a designated parameters cell and writing the executed notebook — with all outputs — to a new file. This makes notebook runs reproducible and self-documenting: the output notebook is a complete record of what ran, with what parameters, and what it produced.

**Parameters cell convention.** papermill locates the injection point by finding a cell tagged `parameters`. In JupyterLab this is done via the cell metadata editor. In raw notebook JSON the cell must have `"tags": ["parameters"]` in its metadata:

```python
# Cell tagged "parameters"
epochs = 10
lr     = 1e-3
model  = "resnet18"
```

**Injection mechanism.** When papermill runs, it inserts a new cell *after* the parameters cell with the runtime overrides:

```python
# Injected parameters
epochs = 20        # overrides the default 10
lr     = 3e-4      # overrides 1e-3
model  = "resnet18" # unchanged — still the default
```

The rest of the notebook uses `epochs`, `lr`, and `model` as ordinary Python variables — they are just in scope by the time execution reaches them.

The papermill call produced by `NotebookRunner` for the spec above:

In [ ]:
spec = JobSpec(
    id="run-42",
    type=JobType.notebook,
    entrypoint="train.ipynb",
    params={"epochs": 20, "lr": 3e-4},
)

cmd = NotebookRunner().build_command(spec)
print(" ".join(cmd))

The output notebook `output_run-42.ipynb` is the artifact we retrieve after the job completes — it contains every cell output, every print statement, every exception traceback. It is the first thing to inspect when a remote run fails.

---

■